In [1]:
# ============================================================
# CREATE 168-HOUR CASE-STUDY HISTORY FILES FOR PLEGMA UI/API EVALUATION
# ============================================================
#
# For each selected PLEGMA home:
#   - creates C:/Plegma_Programming/New_Evaluation/Histories168/<home_id>/
#   - saves UI-ready history_store.csv with columns:
#       timestamp, consumption_Wh
#   - saves actual_target_day.csv for later evaluation
#   - saves case_metadata.json
#   - saves global evaluation_cases_summary.csv
#
# Important:
#   history window is strictly before target_date:
#   [target_date - 168 hours, target_date)
#
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np


# ============================================================
# CONFIG
# ============================================================

# Change this path if your final PLEGMA hourly dataset has another name/location
DATA_PATH = Path(r"C:/Plegma_Programming/PLEGMA_final_hourly_dataset.csv")

OUT_DIR = Path(r"C:/Plegma_Programming/New_Evaluation/SEASONAL")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

HISTORY_HOURS = 168
HISTORY_DAYS = HISTORY_HOURS // 24

EXPECTED_HISTORY_HOURS = 168
EXPECTED_TARGET_HOURS = 24

SAVE_ENCODING = "utf-8-sig"

# Selected PLEGMA case-study homes and dates
CASES = [
    {"home_id": "home01", "target_date": "2022-08-13"},
    {"home_id": "home04", "target_date": "2022-10-23"},
    {"home_id": "home07", "target_date": "2023-05-02"},
    {"home_id": "home05", "target_date": "2023-02-15"},
]


# ============================================================
# HELPERS
# ============================================================

def normalize_home_id(x):
    """
    Normalize different home-id formats:
      House_01 -> home01
      house_01 -> home01
      Home_01  -> home01
      home01   -> home01
    """
    if pd.isna(x):
        return x

    s = str(x).strip()

    if s.lower().startswith("house_"):
        num = s.split("_")[-1]
        return f"home{int(num):02d}"

    if s.lower().startswith("home_"):
        num = s.split("_")[-1]
        return f"home{int(num):02d}"

    if s.lower().startswith("home"):
        suffix = s.lower().replace("home", "").replace("_", "")
        if suffix.isdigit():
            return f"home{int(suffix):02d}"

    return s


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tries to standardize common column names to:
      timestamp, home_id, consumption_Wh

    If your dataset already has these names, nothing changes.
    """

    col_map = {}

    lower_cols = {c.lower(): c for c in df.columns}

    # timestamp column
    possible_time_cols = [
        "timestamp", "datetime", "date_time", "time", "date"
    ]

    for c in possible_time_cols:
        if c in lower_cols:
            col_map[lower_cols[c]] = TIME_COL
            break

    # home_id column
    possible_home_cols = [
        "home_id", "house_id", "house", "home"
    ]

    for c in possible_home_cols:
        if c in lower_cols:
            col_map[lower_cols[c]] = HOME_COL
            break

    # consumption column
    possible_target_cols = [
        "consumption_wh",
        "energy_wh",
        "total_consumption_wh",
        "total_energy_wh",
        "consumption",
        "energy"
    ]

    for c in possible_target_cols:
        if c in lower_cols:
            col_map[lower_cols[c]] = TARGET_COL
            break

    df = df.rename(columns=col_map)

    required = [TIME_COL, HOME_COL, TARGET_COL]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"Missing required columns after standardization: {missing}\n"
            f"Available columns: {list(df.columns)}\n"
            f"Please check DATA_PATH or adjust TIME_COL/HOME_COL/TARGET_COL."
        )

    return df


def ensure_hourly_unique(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure one row per home/timestamp.

    If duplicates exist, average consumption_Wh to avoid double counting.
    """

    tmp = df[[HOME_COL, TIME_COL, TARGET_COL]].copy()

    tmp[TIME_COL] = pd.to_datetime(tmp[TIME_COL], errors="coerce")
    tmp[TARGET_COL] = pd.to_numeric(tmp[TARGET_COL], errors="coerce")

    tmp = tmp.dropna(subset=[HOME_COL, TIME_COL, TARGET_COL])

    tmp[HOME_COL] = tmp[HOME_COL].apply(normalize_home_id)

    tmp = (
        tmp
        .groupby([HOME_COL, TIME_COL], as_index=False)
        .agg({TARGET_COL: "mean"})
    )

    return tmp


def make_complete_hour_index(start_ts: pd.Timestamp, end_exclusive: pd.Timestamp):
    return pd.date_range(
        start_ts,
        end_exclusive - pd.Timedelta(hours=1),
        freq="h"
    )


def add_missing_hour_check(
    data: pd.DataFrame,
    start_ts: pd.Timestamp,
    end_exclusive: pd.Timestamp
):
    expected_index = make_complete_hour_index(start_ts, end_exclusive)
    actual_index = pd.DatetimeIndex(pd.to_datetime(data[TIME_COL]))

    missing = expected_index.difference(actual_index)
    extra = actual_index.difference(expected_index)

    return {
        "expected_hours": len(expected_index),
        "actual_rows": len(data),
        "missing_hours_count": len(missing),
        "extra_hours_count": len(extra),
        "missing_hours": [str(x) for x in missing[:50]],
        "extra_hours": [str(x) for x in extra[:50]],
    }


# ============================================================
# LOAD DATA
# ============================================================

print("=" * 120)
print("Loading PLEGMA final hourly dataset")
print("=" * 120)

df = pd.read_csv(DATA_PATH, low_memory=False)
df = standardize_columns(df)
df = ensure_hourly_unique(df)

print("Rows:", len(df))
print("Homes:", df[HOME_COL].nunique())
print("Available homes:", sorted(df[HOME_COL].unique()))
print("Date range:", df[TIME_COL].min(), "to", df[TIME_COL].max())

summary_rows = []


# ============================================================
# CREATE FILES PER CASE
# ============================================================

for case in CASES:
    home_id = normalize_home_id(case["home_id"])
    target_date = pd.Timestamp(case["target_date"])

    target_start = target_date
    target_end = target_start + pd.Timedelta(days=1)

    history_start = target_start - pd.Timedelta(hours=HISTORY_HOURS)
    history_end = target_start

    home_dir = OUT_DIR / home_id
    home_dir.mkdir(parents=True, exist_ok=True)

    home_df = df[df[HOME_COL] == home_id].copy()

    if home_df.empty:
        print(f"[WARN] {home_id}: no rows found in dataset.")
        continue

    history_df = home_df[
        (home_df[TIME_COL] >= history_start) &
        (home_df[TIME_COL] < history_end)
    ].copy()

    target_df = home_df[
        (home_df[TIME_COL] >= target_start) &
        (home_df[TIME_COL] < target_end)
    ].copy()

    history_df = history_df.sort_values(TIME_COL).reset_index(drop=True)
    target_df = target_df.sort_values(TIME_COL).reset_index(drop=True)

    # UI/API-ready history file: only required columns
    history_out = history_df[[TIME_COL, TARGET_COL]].copy()
    history_out[TIME_COL] = history_out[TIME_COL].dt.strftime("%Y-%m-%d %H:%M:%S")

    # Actual target day for later evaluation
    target_out = target_df[[TIME_COL, TARGET_COL]].copy()
    target_out[TIME_COL] = target_out[TIME_COL].dt.strftime("%Y-%m-%d %H:%M:%S")

    history_file = home_dir / "history_store.csv"
    target_file = home_dir / "actual_target_day.csv"

    history_out.to_csv(history_file, index=False, encoding=SAVE_ENCODING)
    target_out.to_csv(target_file, index=False, encoding=SAVE_ENCODING)

    history_check = add_missing_hour_check(history_df, history_start, history_end)
    target_check = add_missing_hour_check(target_df, target_start, target_end)

    history_total_kWh = float(history_df[TARGET_COL].sum() / 1000.0) if len(history_df) else np.nan
    target_total_kWh = float(target_df[TARGET_COL].sum() / 1000.0) if len(target_df) else np.nan

    metadata = {
        "home_id": home_id,
        "target_date": target_start.date().isoformat(),

        "history_hours": HISTORY_HOURS,
        "history_days": HISTORY_DAYS,

        "history_start": str(history_start),
        "history_end_exclusive": str(history_end),

        "target_start": str(target_start),
        "target_end_exclusive": str(target_end),

        "history_file": str(history_file),
        "target_actual_file": str(target_file),

        "history_expected_hours": EXPECTED_HISTORY_HOURS,
        "history_rows": int(len(history_out)),
        "history_missing_hours_count": int(history_check["missing_hours_count"]),
        "history_complete": bool(
            len(history_out) == EXPECTED_HISTORY_HOURS and
            history_check["missing_hours_count"] == 0
        ),

        "target_expected_hours": EXPECTED_TARGET_HOURS,
        "target_rows": int(len(target_out)),
        "target_missing_hours_count": int(target_check["missing_hours_count"]),
        "target_complete": bool(
            len(target_out) == EXPECTED_TARGET_HOURS and
            target_check["missing_hours_count"] == 0
        ),

        "history_total_kWh": history_total_kWh,
        "history_mean_daily_kWh": float(history_total_kWh / HISTORY_DAYS) if len(history_df) else np.nan,
        "history_mean_Wh": float(history_df[TARGET_COL].mean()) if len(history_df) else np.nan,
        "history_median_Wh": float(history_df[TARGET_COL].median()) if len(history_df) else np.nan,

        "target_total_kWh": target_total_kWh,
        "target_mean_Wh": float(target_df[TARGET_COL].mean()) if len(target_df) else np.nan,
        "target_median_Wh": float(target_df[TARGET_COL].median()) if len(target_df) else np.nan,

        "history_missing_hours_preview": history_check["missing_hours"],
        "target_missing_hours_preview": target_check["missing_hours"],
    }

    metadata_file = home_dir / "case_metadata.json"

    with open(metadata_file, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    summary_rows.append({
        "home_id": home_id,
        "target_date": target_start.date().isoformat(),

        "history_start": history_start,
        "history_end_exclusive": history_end,

        "history_rows": len(history_out),
        "history_expected_hours": EXPECTED_HISTORY_HOURS,
        "history_missing_hours": history_check["missing_hours_count"],
        "history_complete": metadata["history_complete"],

        "target_rows": len(target_out),
        "target_expected_hours": EXPECTED_TARGET_HOURS,
        "target_missing_hours": target_check["missing_hours_count"],
        "target_complete": metadata["target_complete"],

        "history_total_kWh": metadata["history_total_kWh"],
        "history_mean_daily_kWh": metadata["history_mean_daily_kWh"],
        "target_total_kWh": metadata["target_total_kWh"],

        "history_file": str(history_file),
        "target_actual_file": str(target_file),
        "metadata_file": str(metadata_file),
    })

    print("=" * 120)
    print(f"{home_id} | target_date={target_start.date()}")
    print("=" * 120)
    print("History:", history_start, "to", history_end, "| rows:", len(history_out), "/", EXPECTED_HISTORY_HOURS)
    print("Target: ", target_start, "to", target_end, "| rows:", len(target_out), "/", EXPECTED_TARGET_HOURS)
    print("History complete:", metadata["history_complete"])
    print("Target complete: ", metadata["target_complete"])
    print("History total kWh:", round(metadata["history_total_kWh"], 4))
    print("History mean daily kWh:", round(metadata["history_mean_daily_kWh"], 4))
    print("Target total kWh:", round(metadata["target_total_kWh"], 4))
    print("Saved history:", history_file)
    print("Saved actual target:", target_file)


# ============================================================
# SAVE GLOBAL SUMMARY
# ============================================================

summary = pd.DataFrame(summary_rows)

summary_file = OUT_DIR / "evaluation_cases_summary.csv"
summary.to_csv(summary_file, index=False, encoding=SAVE_ENCODING)

cases_file = OUT_DIR / "selected_case_study_homes_and_dates.csv"
pd.DataFrame(CASES).to_csv(cases_file, index=False, encoding=SAVE_ENCODING)

print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)

if not summary.empty:
    print(
        summary[
            [
                "home_id",
                "target_date",
                "history_rows",
                "history_expected_hours",
                "history_missing_hours",
                "history_complete",
                "target_rows",
                "target_expected_hours",
                "target_missing_hours",
                "target_complete",
                "history_total_kWh",
                "history_mean_daily_kWh",
                "target_total_kWh",
            ]
        ].to_string(index=False)
    )
else:
    print("[WARN] No summary rows were created.")

print("=" * 120)
print("Saved")
print("=" * 120)
print("Summary:", summary_file)
print("Cases:", cases_file)
print("Root output folder:", OUT_DIR)

Loading PLEGMA final hourly dataset
Rows: 71111
Homes: 10
Available homes: ['home01', 'home02', 'home03', 'home04', 'home05', 'home07', 'home09', 'home10', 'home11', 'home13']
Date range: 2022-07-11 14:00:00 to 2023-10-01 01:00:00
home01 | target_date=2022-08-13
History: 2022-08-06 00:00:00 to 2022-08-13 00:00:00 | rows: 168 / 168
Target:  2022-08-13 00:00:00 to 2022-08-14 00:00:00 | rows: 24 / 24
History complete: True
Target complete:  True
History total kWh: 17.4518
History mean daily kWh: 2.4931
Target total kWh: 2.4939
Saved history: C:\Plegma_Programming\New_Evaluation\SEASONAL\home01\history_store.csv
Saved actual target: C:\Plegma_Programming\New_Evaluation\SEASONAL\home01\actual_target_day.csv
home04 | target_date=2022-10-23
History: 2022-10-16 00:00:00 to 2022-10-23 00:00:00 | rows: 168 / 168
Target:  2022-10-23 00:00:00 to 2022-10-24 00:00:00 | rows: 24 / 24
History complete: True
Target complete:  True
History total kWh: 30.0596
History mean daily kWh: 4.2942
Target total k